In [1]:
################################################################################
# 🧩 Analisis_Exp_Def_Full — Versión Ultra Optimizada para RAM Limitada
# Fecha: 2025-11-10
################################################################################

import os
import gc
import pandas as pd
import numpy as np
from tqdm import tqdm
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

# =============================================================================
# 1️⃣ LIMPIEZA DE MEMORIA AGRESIVA
# =============================================================================
def limpiar_memoria():
    """Libera memoria de objetos temporales de forma agresiva"""
    gc.collect()
    gc.collect()  # Doble llamada para asegurar limpieza
    

def limpiar_memoria_verbose():
    """Limpieza con reporte de memoria"""
    import psutil
    proceso = psutil.Process(os.getpid())
    mem_antes = proceso.memory_info().rss / (1024 * 1024)
    
    gc.collect()
    gc.collect()
    
    mem_despues = proceso.memory_info().rss / (1024 * 1024)
    print(f"🧹 Memoria: {mem_antes:.1f} MB → {mem_despues:.1f} MB (liberados: {mem_antes - mem_despues:.1f} MB)")

In [2]:
# =============================================================================
# 2️⃣ FUNCIÓN: CARGAR DICCIONARIO DE HOMOLOGACIÓN
# =============================================================================
def cargar_diccionario_homologacion(ruta_excel, hoja):
    """Carga el diccionario de homologación de campos."""
    try:
        df = pd.read_excel(ruta_excel, sheet_name=hoja)
        id_estandar = df.columns[0]
        print(f"✅ Diccionario cargado con {len(df)} filas y ID estándar: {id_estandar}")
        return df, id_estandar
    except FileNotFoundError:
        print(f"💥 Error: Archivo no encontrado en la ruta: {ruta_excel}")
        return None, None
    except Exception as e:
        print(f"💥 Error cargando el diccionario: {e}")
        return None, None

In [3]:
# =============================================================================
# 3️⃣ FUNCIÓN: LISTAR ARCHIVOS (SIN CARGAR EN MEMORIA)
# =============================================================================
def listar_archivos_procesados(ruta_base):
    """
    Lista todos los archivos parquet válidos SIN cargarlos en memoria.
    Solo retorna las rutas de los archivos.
    """
    print("\n📁 Buscando archivos en:", ruta_base)
    
    if not os.path.exists(ruta_base):
        print(f"💥 Error: Carpeta no encontrada en la ruta: {ruta_base}")
        return []
    
    try:
        archivos = sorted([
            os.path.join(ruta_base, f)
            for f in os.listdir(ruta_base)
            if f.startswith("defunciones_") and f.endswith("_procesado.parquet")
        ])
    except Exception as e:
        print(f"💥 Error al listar archivos en {ruta_base}: {e}")
        return []

    if not archivos:
        print("⚠️ No se encontraron archivos para procesar.")
        print(f"   Buscando archivos que cumplan: defunciones_*_procesado.parquet")
        return []

    print(f"✅ Encontrados {len(archivos)} archivos:")
    for i, archivo in enumerate(archivos, 1):
        try:
            # Solo leer metadatos para verificar el archivo
            metadata = pq.read_metadata(archivo)
            num_filas = metadata.num_rows
            print(f"   [{i:02d}] {os.path.basename(archivo)} — {num_filas:,} filas")
        except Exception as e:
            print(f"   [{i:02d}] {os.path.basename(archivo)} — Error al leer metadatos: {e}")
    
    return archivos

In [4]:
# =============================================================================
# 4️⃣ FUNCIÓN: IDENTIFICAR COLUMNAS ÚNICAS (SIN CARGAR DATOS)
# =============================================================================
def identificar_columnas_globales(lista_rutas):
    """
    Identifica todas las columnas únicas leyendo solo los esquemas,
    sin cargar los datos completos.
    """
    print("\n🔍 Identificando columnas únicas (solo esquemas)...")
    todas_columnas = set()
    
    for ruta in lista_rutas:
        try:
            # Leer solo el esquema, no los datos
            schema = pq.read_schema(ruta)
            columnas = schema.names
            todas_columnas.update(columnas)
        except Exception as e:
            print(f"⚠️ Error leyendo esquema de {os.path.basename(ruta)}: {e}")
    
    todas_columnas = sorted(list(todas_columnas))
    print(f"✅ Total de columnas únicas encontradas: {len(todas_columnas)}")
    
    if len(todas_columnas) > 0:
        print(f"   Primeras 10: {todas_columnas[:10]}")
        print(f"   Últimas 5: {todas_columnas[-5:]}")
    
    return todas_columnas

In [5]:
# =============================================================================
# 5️⃣ FUNCIÓN: PROCESAR UN SOLO ARCHIVO
# =============================================================================
def procesar_archivo_individual(ruta_archivo, todas_columnas, carpeta_temp, indice):
    """
    Procesa un solo archivo:
    1. Lee el archivo
    2. Normaliza columnas
    3. Convierte a string
    4. Guarda temporal
    5. Libera memoria
    
    Retorna: (ruta_temp, num_filas, exito)
    """
    nombre_archivo = os.path.basename(ruta_archivo)
    
    try:
        # 🔸 Leer archivo (chunked si es muy grande)
        df = pd.read_parquet(ruta_archivo)
        filas_originales = len(df)
        
        # 🔸 Agregar columnas faltantes
        columnas_faltantes = set(todas_columnas) - set(df.columns)
        if columnas_faltantes:
            for col in columnas_faltantes:
                df[col] = ""
        
        # 🔸 Reordenar columnas
        df = df[todas_columnas]
        
        # 🔸 Convertir a string de forma eficiente
        # Procesar por lotes de columnas para no saturar memoria
        COLUMNAS_POR_LOTE = 50
        for i in range(0, len(todas_columnas), COLUMNAS_POR_LOTE):
            cols_lote = todas_columnas[i:i+COLUMNAS_POR_LOTE]
            for col in cols_lote:
                try:
                    df[col] = df[col].fillna("").astype(str)
                except Exception as e:
                    print(f"⚠️ Error en columna {col}: {e}")
                    df[col] = ""
        
        # 🔸 Guardar temporal
        nombre_temp = f"parte_{indice:03d}.parquet"
        ruta_temp = carpeta_temp / nombre_temp
        
        df.to_parquet(ruta_temp, index=False, engine='pyarrow', compression='snappy')
        
        # 🔸 Liberar memoria INMEDIATAMENTE
        del df
        limpiar_memoria()
        
        return (ruta_temp, filas_originales, True)
        
    except Exception as e:
        print(f"💥 Error procesando {nombre_archivo}: {e}")
        limpiar_memoria()
        return (None, 0, False)

In [6]:
# =============================================================================
# 6️⃣ FUNCIÓN: COMBINACIÓN ULTRA OPTIMIZADA
# =============================================================================
def combinacion_ultra_optimizada(lista_rutas, ruta_salida_final, todas_columnas):
    """
    Combina archivos procesando UNO A LA VEZ para minimizar uso de RAM.
    """
    
    # Carpeta temporal
    carpeta_temp = Path("data/combined_temp")
    carpeta_temp.mkdir(parents=True, exist_ok=True)

    total_filas = 0
    partes_guardadas = []
    archivos_exitosos = 0

    print("\n🔄 Procesando archivos UNO POR UNO (ultra optimizado)...\n")

    # ==========================================================
    # 🔹 PROCESAR CADA ARCHIVO INDIVIDUALMENTE
    # ==========================================================
    for i, ruta in enumerate(tqdm(lista_rutas, desc="Procesando archivos")):
        nombre = os.path.basename(ruta)
        
        print(f"\n📦 [{i+1}/{len(lista_rutas)}] Procesando: {nombre}")
        
        ruta_temp, num_filas, exito = procesar_archivo_individual(
            ruta, todas_columnas, carpeta_temp, i+1
        )
        
        if exito and ruta_temp:
            partes_guardadas.append(ruta_temp)
            total_filas += num_filas
            archivos_exitosos += 1
            print(f"   ✅ Guardado temporal — Filas acumuladas: {total_filas:,}")
        else:
            print(f"   ❌ Falló el procesamiento")
        
        # Limpieza agresiva después de cada archivo
        limpiar_memoria_verbose()

    print(f"\n✅ Archivos procesados exitosamente: {archivos_exitosos}/{len(lista_rutas)}")

    if not partes_guardadas:
        print("❌ No hay archivos temporales para consolidar.")
        return

    # ==========================================================
    # 🔹 CONSOLIDAR USANDO PYARROW (MÁS EFICIENTE)
    # ==========================================================
    print("\n🧩 Consolidando archivos temporales...")
    print("   (Esto puede tomar varios minutos dependiendo del tamaño)")
    
    try:
        # Leer todos los archivos temporales como tablas de Arrow
        tablas = []
        for i, archivo in enumerate(tqdm(partes_guardadas, desc="Leyendo temporales")):
            try:
                tabla = pq.read_table(archivo)
                tablas.append(tabla)
                
                # Limpieza cada 10 archivos
                if (i + 1) % 10 == 0:
                    limpiar_memoria()
                    
            except Exception as e:
                print(f"💥 Error leyendo {archivo}: {e}")

        if not tablas:
            print("❌ No se pudieron leer las tablas temporales.")
            return

        # Concatenar
        print("🔗 Concatenando tablas...")
        tabla_final = pa.concat_tables(tablas, promote=True)
        
        # Liberar memoria de tablas individuales
        del tablas
        limpiar_memoria_verbose()
        
        print(f"✅ Concatenación exitosa: {tabla_final.num_rows:,} filas")

        # Guardar
        print(f"💾 Guardando archivo final (esto puede tardar)...")
        pq.write_table(
            tabla_final, 
            ruta_salida_final,
            compression='snappy'  # Buena compresión sin mucho overhead
        )
        
        del tabla_final
        limpiar_memoria()
        
        # Verificar resultado
        tamaño_mb = os.path.getsize(ruta_salida_final) / (1024 * 1024)
        print(f"\n✅ ARCHIVO FINAL CREADO:")
        print(f"   📂 Ruta: {ruta_salida_final}")
        print(f"   📊 Filas: {total_filas:,}")
        print(f"   📊 Columnas: {len(todas_columnas)}")
        print(f"   📊 Tamaño: {tamaño_mb:.2f} MB")
        
    except Exception as e:
        print(f"💥 Error durante la consolidación: {e}")
        import traceback
        traceback.print_exc()
        return

    # ==========================================================
    # 🔹 LIMPIAR ARCHIVOS TEMPORALES
    # ==========================================================
    print("\n🗑️ Limpiando archivos temporales...")
    try:
        archivos_eliminados = 0
        for archivo in partes_guardadas:
            if os.path.exists(archivo):
                os.remove(archivo)
                archivos_eliminados += 1
        
        print(f"   ✅ Eliminados {archivos_eliminados} archivos temporales")
        
        # Eliminar carpeta si está vacía
        if os.path.exists(carpeta_temp) and not os.listdir(carpeta_temp):
            os.rmdir(carpeta_temp)
            print(f"   ✅ Carpeta temporal eliminada")
        
    except Exception as e:
        print(f"⚠️ Error limpiando temporales: {e}")

    limpiar_memoria_verbose()

In [7]:
# =============================================================================
# 7️⃣ FUNCIÓN PRINCIPAL - ULTRA OPTIMIZADA
# =============================================================================
def main():
    print("="*80)
    print("🚀 PROCESO DE COMBINACIÓN — MODO ULTRA OPTIMIZADO PARA RAM LIMITADA")
    print("="*80)

    try:
        import psutil
        proceso = psutil.Process(os.getpid())
        mem_disponible = psutil.virtual_memory().available / (1024 * 1024 * 1024)
        print(f"💾 RAM disponible: {mem_disponible:.2f} GB\n")
    except:
        pass

    # 1️⃣ Definir rutas
    base_dir = "data"
    processed_data_path = os.path.join(base_dir, "processed")
    output_dir = os.path.join(base_dir, "processed")
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"📂 Ruta de entrada: {processed_data_path}")
    print(f"📂 Ruta de salida: {output_dir}\n")

    # 2️⃣ Listar archivos (SIN cargarlos)
    lista_rutas = listar_archivos_procesados(processed_data_path)
    
    if not lista_rutas:
        print("\n❌ No hay archivos para procesar.")
        return

    # 3️⃣ Identificar columnas (solo esquemas)
    todas_columnas = identificar_columnas_globales(lista_rutas)
    
    if not todas_columnas:
        print("❌ No se pudieron identificar columnas.")
        return

    # 4️⃣ Combinar archivos
    final_output_path = os.path.join(output_dir, "defunciones.parquet")
    
    print(f"\n🎯 Archivo final: {final_output_path}\n")
    
    combinacion_ultra_optimizada(
        lista_rutas,
        final_output_path,
        todas_columnas
    )
    
    # 5️⃣ Verificación final ligera
    if os.path.exists(final_output_path):
        print("\n" + "="*80)
        print("✅ PROCESO COMPLETADO CON ÉXITO")
        print("="*80)
        
        try:
            # Leer solo metadatos para verificar
            metadata = pq.read_metadata(final_output_path)
            print(f"\n📊 VERIFICACIÓN FINAL:")
            print(f"   • Filas totales: {metadata.num_rows:,}")
            print(f"   • Columnas: {metadata.num_columns}")
            
            # Leer solo las primeras 3 filas para muestra
            print(f"\n📄 Muestra (primeras 3 filas):")
            df_muestra = pd.read_parquet(final_output_path, nrows=3)
            print(df_muestra)
            del df_muestra
            
        except Exception as e:
            print(f"⚠️ No se pudo verificar el archivo: {e}")
    else:
        print("\n❌ El archivo final no se creó.")

    limpiar_memoria_verbose()
    print("\n" + "="*80)
    print("🏁 PROCESO FINALIZADO")
    print("="*80)

In [8]:
# =============================================================================
# 8️⃣ EJECUCIÓN
# =============================================================================
if __name__ == "__main__":
    main()

🚀 PROCESO DE COMBINACIÓN — MODO ULTRA OPTIMIZADO PARA RAM LIMITADA
💾 RAM disponible: 15.75 GB

📂 Ruta de entrada: data\processed
📂 Ruta de salida: data\processed


📁 Buscando archivos en: data\processed
✅ Encontrados 17 archivos:
   [01] defunciones_1979_1991_procesado.parquet — 1,869,025 filas
   [02] defunciones_1992_1996_procesado.parquet — 848,360 filas
   [03] defunciones_1997_1997_procesado.parquet — 170,753 filas
   [04] defunciones_1998_2007_procesado.parquet — 1,886,949 filas
   [05] defunciones_2008_2011_procesado.parquet — 790,223 filas
   [06] defunciones_2012_2013_procesado.parquet — 402,827 filas
   [07] defunciones_2014_procesado.parquet — 210,051 filas
   [08] defunciones_2015_procesado.parquet — 219,472 filas
   [09] defunciones_2016_procesado.parquet — 223,078 filas
   [10] defunciones_2017_procesado.parquet — 227,624 filas
   [11] defunciones_2018_procesado.parquet — 236,932 filas
   [12] defunciones_2019_procesado.parquet — 244,355 filas
   [13] defunciones_2020_pro

Procesando archivos:   0%|          | 0/17 [00:00<?, ?it/s]


📦 [1/17] Procesando: defunciones_1979_1991_procesado.parquet


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_18004\2612400279.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = ""
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_18004\2612400279.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = ""
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_18004\2612400279.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

   ✅ Guardado temporal — Filas acumuladas: 1,869,025
🧹 Memoria: 1295.8 MB → 1295.8 MB (liberados: 0.0 MB)

📦 [2/17] Procesando: defunciones_1992_1996_procesado.parquet


...

   ✅ Guardado temporal — Filas acumuladas: 8,549,253
🧹 Memoria: 564.7 MB → 564.7 MB (liberados: 0.0 MB)

📦 [17/17] Procesando: defunciones_2024_procesado.parquet


Procesando archivos: 100%|██████████| 17/17 [04:29<00:00, 15.86s/it]


   ✅ Guardado temporal — Filas acumuladas: 8,825,031
🧹 Memoria: 572.1 MB → 572.1 MB (liberados: 0.0 MB)

✅ Archivos procesados exitosamente: 17/17

🧩 Consolidando archivos temporales...
   (Esto puede tomar varios minutos dependiendo del tamaño)


Leyendo temporales: 100%|██████████| 17/17 [00:11<00:00,  1.52it/s]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_18004\3000186280.py:45: FutureWarning: promote has been superseded by promote_options='default'.
  combinacion_ultra_optimizada(


🔗 Concatenando tablas...
🧹 Memoria: 9318.1 MB → 9316.1 MB (liberados: 2.0 MB)
✅ Concatenación exitosa: 8,825,031 filas
💾 Guardando archivo final (esto puede tardar)...

✅ ARCHIVO FINAL CREADO:
   📂 Ruta: data\processed\defunciones.parquet
   📊 Filas: 8,825,031
   📊 Columnas: 137
   📊 Tamaño: 262.43 MB

🗑️ Limpiando archivos temporales...
   ✅ Eliminados 17 archivos temporales
   ✅ Carpeta temporal eliminada
🧹 Memoria: 8567.1 MB → 8567.1 MB (liberados: 0.0 MB)

✅ PROCESO COMPLETADO CON ÉXITO

📊 VERIFICACIÓN FINAL:
   • Filas totales: 8,825,031
   • Columnas: 137

📄 Muestra (primeras 3 filas):
⚠️ No se pudo verificar el archivo: read_table() got an unexpected keyword argument 'nrows'
🧹 Memoria: 8565.0 MB → 8565.0 MB (liberados: 0.0 MB)

🏁 PROCESO FINALIZADO
